In [ ]:
import pandas as pd 
import os 
import sys

sys.path.append("../src")
from llm_client import call_llm
from prompt_templates import *

c:\Stellantis\HomeNestReviews_llm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Stellantis\HomeNestReviews_llm\notebooks\../src\llm_client.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [7]:
df = pd.read_csv("../data/raw/homenest_reviews.csv")
df.head()

,review_id,product_category,product_name,review_text,star_rating,review_date,verified_purchase
0,1,Bedroom Furniture,DreamRest Memory Foam Mattress,My DreamRest Memory Foam Mattress arrived well...,5,2024-02-08,False
1,2,Kitchen Appliances,InstaBrew Coffee Maker,The InstaBrew Coffee Maker itself is fine but ...,2,2023-09-11,False
2,3,Living Room,GlowLight Floor Lamp,My GlowLight Floor Lamp arrived well packaged ...,5,2023-07-03,False
3,4,Living Room,CozySofa L-Shape Sectional,The CozySofa L-Shape Sectional is okay. Does w...,3,2023-06-30,True
4,5,Kitchen Appliances,TurboBlend Pro 5000,Delivery took 3 weeks and the TurboBlend Pro 5...,3,2024-09-23,True


In [11]:
def rating_to_sentiment(star):
    if star > 4 :
        return "Positive"
    elif star == 3:
        return "Neutral"
    else :
        return "Negative"
    


df["ground_truth"] = df["star_rating"].apply(rating_to_sentiment)

Zero -Shot and Few Shot prompts

In [8]:
ZERO_SHOT_SYSTEM = """
You are a sentiment classifier.

Respond ONLY with:
Positive
Negative
Neutral
"""

In [9]:
FEW_SHOT_SYSTEM = "You are a sentiment classifier."

In [24]:
results = []

for i,row in df.head(200).iterrows():

    review = row["review_text"]
    truth = row["ground_truth"]

    zero = call_llm(ZERO_SHOT_SYSTEM, f"Review: {review}", provider="mistral")

    few_prompt = f"""
Classify the sentiment of customer reviews.

Examples:
Review: 'Absolutely love this blender! Works perfectly.' -> Positive
Review: 'Broke after 2 weeks. Total waste of money.' -> Negative
Review: 'It is okay. Does what it says, nothing special.' -> Neutral

Now classify:

Review: '{review}' ->
"""

    few = call_llm(FEW_SHOT_SYSTEM, few_prompt, provider="mistral")

    results.append({
        "review_id": i,
        "ground_truth": truth,
        "zero_shot": zero.strip(),
        "few_shot": few.strip()
    })

In [26]:
print(results)

[{'review_id': 0, 'ground_truth': 'Positive', 'zero_shot': 'Positive', 'few_shot': 'Positive'}, {'review_id': 1, 'ground_truth': 'Negative', 'zero_shot': 'Neutral', 'few_shot': 'Neutral (leaning slightly Negative)\n\nThe review expresses a mixed sentiment. The customer states that the product itself is "fine," which is a neutral to slightly positive statement. However, they also mention that the packaging was "completely destroyed," which introduces a negative aspect. Since the negative part does not directly criticize the product\'s functionality but rather an external factor (packaging), the overall sentiment is best classified as **Neutral**, though it leans slightly negative due to the frustration implied.'}, {'review_id': 2, 'ground_truth': 'Positive', 'zero_shot': 'Positive', 'few_shot': 'Positive'}, {'review_id': 3, 'ground_truth': 'Neutral', 'zero_shot': 'Neutral', 'few_shot': 'Neutral'}, {'review_id': 4, 'ground_truth': 'Neutral', 'zero_shot': 'Negative', 'few_shot': "Negative

In [27]:
results_df = pd.DataFrame(results)
results_df

,review_id,ground_truth,zero_shot,few_shot
0,0,Positive,Positive,Positive
1,1,Negative,Neutral,Neutral (leaning slightly Negative)\n\nThe rev...
2,2,Positive,Positive,Positive
3,3,Neutral,Neutral,Neutral
4,4,Neutral,Negative,Negative\n\nThe review expresses dissatisfacti...
...,...,...,...,...
195,195,Positive,Positive,Positive
196,196,Positive,Positive,Positive
197,197,Positive,Positive,Positive
198,198,Negative,Positive,Positive


In [28]:
results_df["zero_correct"] = results_df["ground_truth"] == results_df["zero_shot"]
results_df["few_correct"] = results_df["ground_truth"] == results_df["few_shot"]

results_df

,review_id,ground_truth,zero_shot,few_shot,zero_correct,few_correct
0,0,Positive,Positive,Positive,True,True
1,1,Negative,Neutral,Neutral (leaning slightly Negative)\n\nThe rev...,False,False
2,2,Positive,Positive,Positive,True,True
3,3,Neutral,Neutral,Neutral,True,True
4,4,Neutral,Negative,Negative\n\nThe review expresses dissatisfacti...,False,False
...,...,...,...,...,...,...
195,195,Positive,Positive,Positive,True,True
196,196,Positive,Positive,Positive,True,True
197,197,Positive,Positive,Positive,True,True
198,198,Negative,Positive,Positive,False,False


In [29]:
print("Zero-shot prompt accuracy :",results_df["zero_correct"].value_counts())
print("\n\nFew-shots prompt accuracy :",results_df["few_correct"].value_counts())

Zero-shot prompt accuracy : zero_correct
True     140
False     60
Name: count, dtype: int64


Few-shots prompt accuracy : few_correct
True     134
False     66
Name: count, dtype: int64


In [30]:
zero_acc = results_df["zero_correct"].mean()
few_acc = results_df["few_correct"].mean()

print("Zero-shot accuracy:", zero_acc)
print("Few-shot accuracy:", few_acc)

Zero-shot accuracy: 0.7
Few-shot accuracy: 0.67


Observation : Mistral is Returning Sentences for few cases instead of Lableing 

Chain of Thought for Nuanced Reviews 

In [61]:
# Find the reviews where Zero-shot Failed 

difficult_reviews = results_df[results_df['zero_correct'] == False]
len(difficult_reviews)

60

In [62]:
difficult_reviews = difficult_reviews.head(10)

In [63]:
difficult_reviews = difficult_reviews.merge(
    df[["review_text"]],
    left_on="review_id",
    right_index=True
)

In [65]:
difficult_reviews

,review_id,ground_truth,zero_shot,few_shot,zero_correct,few_correct,review_text
1,1,Negative,Neutral,Neutral (leaning slightly Negative)\n\nThe rev...,False,False,The InstaBrew Coffee Maker itself is fine but ...
4,4,Neutral,Negative,Negative\n\nThe review expresses dissatisfacti...,False,False,Delivery took 3 weeks and the TurboBlend Pro 5...
7,7,Negative,Positive,Positive,False,False,ABSOLUTELY LOVE MY DREAMREST MEMORY FOAM MATTR...
9,9,Positive,Negative,Negative,False,False,<p>Trying to return my FoamPure Electric Tooth...
17,17,Negative,Positive,Positive,False,False,Bought the AquaFlow Rainfall Showerhead as a g...
18,18,Negative,Positive,Positive,False,False,Five stars. The AromaSpa Diffuser is well buil...
23,23,Negative,Positive,Positive,False,False,"THE GLOWLIGHT FLOOR LAMP IS EASY TO CLEAN, QUI..."
24,24,Negative,Neutral,Neutral,False,False,The CrispAir Fryer XL is okay. Does what it sa...
27,27,Negative,Positive,Positive,False,False,Outstanding product. The InstaBrew Coffee Make...
30,30,Negative,Positive,Positive,False,False,Outstanding product. The SmartTV 55in 4K UHD h...


In [66]:
COT_SYSTEM = """You are an expert review analyst.
 Think carefully before classifying."""


In [67]:
cot_results = []

for i, row in difficult_reviews.iterrows():
    review = row["review_text"]

    prompt = f""" 
    Analyze this customer review step by step:
 1. What is the customer's main experience (positive or negative)?
 2. Are there any contradictions or mixed feelings?
 3. What is the dominant sentiment overall?
 
 Review : '{review}'
 
 Respond in this format:
 Step 1: [your analysis]
 Step 2: [your analysis]
 Step 3: [your analysis]
 Classification: [Positive/Negative/Neutral """

In [68]:
cot_output = call_llm(COT_SYSTEM , prompt , provider= "mistral")
cot_results.append({
    "review_text" : review,
    "cot_output" : cot_output
})

In [69]:
cot_df = pd.DataFrame(cot_results)

print(len(cot_df))

1


In [70]:
import re

def extract_label(text):
    
    match = re.search(r'Classification:\s*(Positive|Negative|Neutral)', text, re.I)
    
    if match:
        return match.group(1).capitalize()
    
    return "Unknown"

In [71]:
cot_df["cot_prediction"] = cot_df["cot_output"].apply(extract_label)

In [74]:
cot_df["ground_truth"] = difficult_reviews["ground_truth"].reset_index(drop=True)

cot_df["cot_correct"] = cot_df["cot_prediction"] == cot_df["ground_truth"]

cot_df

,review_text,cot_output,cot_prediction,ground_truth,cot_correct
0,Outstanding product. The SmartTV 55in 4K UHD h...,Step 1: The customer's main experience is **po...,Positive,Negative,False


JSON output

In [76]:
JSON_SYSTEM = """
You are a senior product review analyst.

You analyze customer reviews for an e-commerce retailer.

You must ALWAYS respond with valid JSON.

Rules:
- Respond ONLY with JSON.
- Do NOT include explanations.
- Do NOT add extra fields.
- Do NOT include text before or after the JSON.
"""

In [77]:
def json_analysis_prompt(review):

    user_prompt = f"""
Analyze the following customer review and return a JSON object with:

sentiment: one of [Positive, Negative, Neutral]
confidence: a float between 0.0 and 1.0
key_issue: the main issue or praise (max 5 words)
is_safety_concern: true if the review mentions fire, smoke, sparks, chemicals, or injury

Review:
"{review}"
"""

    return user_prompt

In [78]:
review = df.iloc[0]["review_text"]

output = call_llm(
    JSON_SYSTEM,
    json_analysis_prompt(review),
    provider="mistral"
)

print(output)

```json
{
  "sentiment": "Positive",
  "confidence": 0.98,
  "key_issue": "no issues, well packaged",
  "is_safety_concern": false
}
```


In [79]:
import json
import re

def safe_parse_json(llm_output):
    """Extract JSON even if the model returns extra text."""

    try:
        return json.loads(llm_output)

    except json.JSONDecodeError:

        match = re.search(r'\{.*?\}', llm_output, re.DOTALL)

        if match:
            return json.loads(match.group())

        return {
            "error": "unparseable",
            "raw": llm_output
        }

In [80]:
parsed = safe_parse_json(output)

parsed

{'sentiment': 'Positive',
 'confidence': 0.98,
 'key_issue': 'no issues, well packaged',
 'is_safety_concern': False}

In [81]:
json_results = []

for review in df["review_text"].head(10):

    output = call_llm(
        JSON_SYSTEM,
        json_analysis_prompt(review),
        provider="mistral"
    )

    parsed = safe_parse_json(output)

    json_results.append(parsed)

In [82]:
json_df = pd.DataFrame(json_results)

json_df

,sentiment,confidence,key_issue,is_safety_concern
0,Positive,0.98,"no issues, well packaged",False
1,Neutral,0.85,packaging damaged on arrival,False
2,Positive,0.98,"No issues, works well",False
3,Neutral,0.90,average quality,False
4,Neutral,0.80,delivery damage and delay,False
5,Positive,0.98,Premium quality great value,False
6,Neutral,0.90,average performance,False
7,Positive,1.00,comfort and delivery,False
8,Positive,0.98,"easy to clean, quiet",False
9,Negative,0.95,poor customer service,False


In [83]:
BAD_SYSTEM = "You are a review analyst."

output = call_llm(
    BAD_SYSTEM,
    json_analysis_prompt(review),
    provider="mistral"
)

print(output)

```json
{
  "sentiment": "Negative",
  "confidence": 0.95,
  "key_issue": "poor customer service",
  "is_safety_concern": false
}
```
